In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Labs"
)

xpt_file = DATA_DIR / "TRIGLY_I.xpt"
csv_file = DATA_DIR / "TRIGLY_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 TRIGLY_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTSAF2YR",
    "LBXTR",
    "LBDTRSI",
    "LBDLDL",
    "LBDLDLSI"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "TRIGLY_I column names/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 3191:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 6:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 3,191 rows x 6 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# TRIGLY_I is a MIXED integer / decimal Laboratory file.
#
#
# SEQN
#     Respondent sequence number
#     -> whole-number ID
#
#
# WTSAF2YR
#     Fasting subsample 2-year MEC weight
#     -> genuine continuous decimal
#
#
# LBXTR
#     Triglycerides mg/dL
#     -> released as whole-number values
#
#
# LBDTRSI
#     Triglycerides mmol/L
#     -> genuine continuous decimal
#
#
# LBDLDL
#     LDL cholesterol mg/dL
#     -> released as whole-number values
#
#
# LBDLDLSI
#     LDL cholesterol mmol/L
#     -> genuine continuous decimal
#
#
# Therefore:
#
#     SEQN
#     LBXTR
#     LBDLDL
#
# can become Int64.
#
#
# DO NOT convert:
#
#     WTSAF2YR
#     LBDTRSI
#     LBDLDLSI
#
# to integers.
#
# DO NOT globally remove ".0".
#

DECIMAL_COLUMNS = [
    "WTSAF2YR",
    "LBDTRSI",
    "LBDLDLSI"
]


INTEGER_COLUMNS = [
    "SEQN",
    "LBXTR",
    "LBDLDL"
]


if len(DECIMAL_COLUMNS) != 3:

    raise ValueError(
        "Expected exactly 3 genuine decimal fields."
    )


if len(INTEGER_COLUMNS) != 3:

    raise ValueError(
        "Expected exactly 3 whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
#
# In the attached TRIGLY_I file this occurs ONLY in:
#
#     WTSAF2YR
#
# and corresponds to legitimate zero fasting weights.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 6 TRIGLY_I fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached TRIGLY_I.xpt:
#
#     WTSAF2YR = 448
#
# TOTAL = 448
#
#
# CDC independently reports:
#
#     positive WTSAF2YR = 2,743
#     WTSAF2YR = 0      =   448
#
# Total                  = 3,191
#

EXPECTED_TINY_COUNTS = {
    "WTSAF2YR": 448
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated TRIGLY_I attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 448:

    raise ValueError(
        f"Unexpected tiny-value total: {tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed "
        "before tiny-value restoration."
    )


print(
    "\nPASS: Exact 448-value WTSAF2YR "
    "tiny-zero pattern validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED WTSAF2YR ZEROS
# ============================================================

#
# NEVER:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

weight_zero_mask = (
    df["WTSAF2YR"]
    .eq(TINY_VALUE)
)


weight_zero_count = int(
    weight_zero_mask.sum()
)


if weight_zero_count != 448:

    raise ValueError(
        "Expected exactly 448 tiny WTSAF2YR values."
    )


df.loc[
    weight_zero_mask,
    "WTSAF2YR"
] = 0


print(
    "\nPASS: 448 validated WTSAF2YR "
    "values restored to numeric zero."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df becomes our exact fidelity reference.
#
# The ONLY allowed numerical correction is:
#
#     WTSAF2YR tiny value -> 0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["WTSAF2YR"].eq(TINY_VALUE),
    "WTSAF2YR"
] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is used ONLY as a diagnostic comparison.
#
# No dataframe values are rounded.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


detected_decimal_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern differs.\n\n"
        f"Expected:\n{DECIMAL_COLUMNS}\n\n"
        f"Actual:\n{detected_decimal_columns}"
    )


print(
    "\nPASS: Exact genuine decimal fields detected:"
)

print(
    DECIMAL_COLUMNS
)


# ============================================================
# 11. DECIMAL VARIABLE DETAILS
# ============================================================

print(
    "\n--- DECIMAL FIELD DETAILS ---"
)


for col in DECIMAL_COLUMNS:

    s = df[col]


    nonmissing = int(
        s.notna()
        .sum()
    )


    missing = int(
        s.isna()
        .sum()
    )


    fractional = fractional_counts[col]


    whole_looking = (
        nonmissing
        -
        fractional
    )


    print(
        f"\n{col}"
    )

    print(
        "  Nonmissing:",
        f"{nonmissing:,}"
    )

    print(
        "  Missing:",
        f"{missing:,}"
    )

    print(
        "  Minimum:",
        repr(float(s.min()))
    )

    print(
        "  Maximum:",
        repr(float(s.max()))
    )

    print(
        "  Fractional observations:",
        f"{fractional:,}"
    )

    print(
        "  Whole-looking observations:",
        f"{whole_looking:,}"
    )


# ============================================================
# 12. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83733:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93697:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83733-93697 validated."
)


# ============================================================
# 13. VALIDATE LBXTR IS WHOLE-NUMBER
# ============================================================

tr_values = (
    df["LBXTR"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    tr_values,
    np.round(tr_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "LBXTR unexpectedly contains fractional values."
    )


print(
    "PASS: All populated LBXTR values "
    "are whole-number."
)


# ============================================================
# 14. VALIDATE LBDLDL IS WHOLE-NUMBER
# ============================================================

ldl_values = (
    df["LBDLDL"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    ldl_values,
    np.round(ldl_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "LBDLDL unexpectedly contains fractional values."
    )


print(
    "PASS: All populated LBDLDL values "
    "are whole-number."
)


# ============================================================
# 15. CONVERT PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# NO round() during actual conversion.
#

df["LBXTR"] = (
    df["LBXTR"]
    .astype("Int64")
)


df["LBDLDL"] = (
    df["LBDLDL"]
    .astype("Int64")
)


print(
    "\nPASS: LBXTR and LBDLDL converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 16. CONFIRM DECIMAL VARIABLES REMAIN FLOAT
# ============================================================

decimal_dtype_failures = []


for col in DECIMAL_COLUMNS:

    if not pd.api.types.is_float_dtype(
        df[col].dtype
    ):

        decimal_dtype_failures.append(
            col
        )


if decimal_dtype_failures:

    raise ValueError(
        "Continuous variables are no longer float:\n"
        + str(
            decimal_dtype_failures
        )
    )


print(
    "PASS: WTSAF2YR, LBDTRSI and LBDLDLSI "
    "remain floating-point."
)


# ============================================================
# 17. CDC / XPT CHECK - WTSAF2YR
# ============================================================

#
# CDC HTML displays:
#
#     13612.331812 to 521632.18583
#
#
# Actual attached XPT stores:
#
#     minimum = 13612.331812498
#     maximum = 521632.1858299
#
#
# Preserve the XPT precision.
#

weight_positive = int(
    df["WTSAF2YR"]
    .gt(0)
    .sum()
)


weight_zero = int(
    df["WTSAF2YR"]
    .eq(0)
    .sum()
)


weight_missing = int(
    df["WTSAF2YR"]
    .isna()
    .sum()
)


positive_weights = (
    df.loc[
        df["WTSAF2YR"] > 0,
        "WTSAF2YR"
    ]
)


weight_min = float(
    positive_weights.min()
)


weight_max = float(
    positive_weights.max()
)


print(
    "\n--- WTSAF2YR ---"
)

print(
    "Positive:",
    f"{weight_positive:,}"
)

print(
    "Zero:",
    f"{weight_zero:,}"
)

print(
    "Missing:",
    f"{weight_missing:,}"
)

print(
    "Exact XPT minimum:",
    repr(weight_min)
)

print(
    "Exact XPT maximum:",
    repr(weight_max)
)


if weight_positive != 2743:

    raise ValueError(
        "WTSAF2YR positive count differs."
    )


if weight_zero != 448:

    raise ValueError(
        "WTSAF2YR zero count differs."
    )


if weight_missing != 0:

    raise ValueError(
        "Unexpected missing WTSAF2YR."
    )


if weight_min != 13612.331812498:

    raise ValueError(
        "WTSAF2YR exact XPT minimum differs."
    )


if weight_max != 521632.1858299:

    raise ValueError(
        "WTSAF2YR exact XPT maximum differs."
    )


#
# Secondary CDC DISPLAY validation only.
#
# Temporary scalars are rounded.
# The dataframe is NEVER rounded.
#

if round(
    weight_min,
    6
) != 13612.331812:

    raise ValueError(
        "WTSAF2YR minimum does not match "
        "CDC display precision."
    )


if round(
    weight_max,
    5
) != 521632.18583:

    raise ValueError(
        "WTSAF2YR maximum does not match "
        "CDC display precision."
    )


print(
    "PASS: WTSAF2YR exact XPT precision preserved."
)


# ============================================================
# 18. CDC CHECK - LBXTR
# ============================================================

#
# CDC:
#
# Triglycerides mg/dL
#
#     14-2141
#
#     nonmissing = 2,723
#     missing    =   468
#

tr_nonmissing = int(
    df["LBXTR"]
    .notna()
    .sum()
)


tr_missing = int(
    df["LBXTR"]
    .isna()
    .sum()
)


tr_range = int(
    df["LBXTR"]
    .between(
        14,
        2141
    )
    .sum()
)


tr_min = int(
    df["LBXTR"]
    .min()
)


tr_max = int(
    df["LBXTR"]
    .max()
)


print(
    "\n--- LBXTR ---"
)

print(
    "Nonmissing:",
    f"{tr_nonmissing:,}"
)

print(
    "Missing:",
    f"{tr_missing:,}"
)

print(
    "Minimum:",
    tr_min
)

print(
    "Maximum:",
    tr_max
)


if tr_nonmissing != 2723:

    raise ValueError(
        "LBXTR nonmissing count differs."
    )


if tr_missing != 468:

    raise ValueError(
        "LBXTR missing count differs."
    )


if tr_range != 2723:

    raise ValueError(
        "LBXTR contains values outside 14-2141."
    )


if tr_min != 14:

    raise ValueError(
        "LBXTR minimum differs."
    )


if tr_max != 2141:

    raise ValueError(
        "LBXTR maximum differs."
    )


print(
    "PASS: LBXTR matches CDC range/count."
)


# ============================================================
# 19. CDC CHECK - LBDTRSI
# ============================================================

#
# Triglycerides mmol/L
#
# CDC:
#
#     0.158-24.172
#
#     nonmissing = 2,723
#     missing    =   468
#

trsi_nonmissing = int(
    df["LBDTRSI"]
    .notna()
    .sum()
)


trsi_missing = int(
    df["LBDTRSI"]
    .isna()
    .sum()
)


trsi_range = int(
    df["LBDTRSI"]
    .between(
        0.158,
        24.172
    )
    .sum()
)


trsi_min = float(
    df["LBDTRSI"]
    .min()
)


trsi_max = float(
    df["LBDTRSI"]
    .max()
)


print(
    "\n--- LBDTRSI ---"
)

print(
    "Nonmissing:",
    f"{trsi_nonmissing:,}"
)

print(
    "Missing:",
    f"{trsi_missing:,}"
)

print(
    "Minimum:",
    repr(trsi_min)
)

print(
    "Maximum:",
    repr(trsi_max)
)


if trsi_nonmissing != 2723:

    raise ValueError(
        "LBDTRSI nonmissing count differs."
    )


if trsi_missing != 468:

    raise ValueError(
        "LBDTRSI missing count differs."
    )


if trsi_range != 2723:

    raise ValueError(
        "LBDTRSI contains values outside CDC range."
    )


if trsi_min != 0.158:

    raise ValueError(
        "LBDTRSI minimum differs."
    )


if trsi_max != 24.172:

    raise ValueError(
        "LBDTRSI maximum differs."
    )


print(
    "PASS: LBDTRSI matches CDC range/count."
)


# ============================================================
# 20. LBXTR / LBDTRSI MISSING-MASK CHECK
# ============================================================

tr_missing_mask = (
    df["LBXTR"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


trsi_missing_mask = (
    df["LBDTRSI"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


tr_mask_mismatches = int(
    np.sum(
        tr_missing_mask
        !=
        trsi_missing_mask
    )
)


print(
    "\n--- TRIGLYCERIDE MISSING MASK ---"
)

print(
    "Mismatches:",
    tr_mask_mismatches
)


if tr_mask_mismatches != 0:

    raise ValueError(
        "LBXTR and LBDTRSI missing positions differ."
    )


print(
    "PASS: LBXTR and LBDTRSI missing "
    "positions match exactly."
)


# ============================================================
# 21. CDC TRIGLYCERIDE SI CONVERSION CHECK
# ============================================================

#
# CDC:
#
#     LBDTRSI = LBXTR * 0.01129
#
#
# Released SI values are represented to 3 decimals.
#
#
# Decimal + ROUND_HALF_UP is used ONLY for validation.
#
# We NEVER overwrite LBDTRSI.
#

valid_tr_mask = (
    df["LBXTR"]
    .notna()
    &
    df["LBDTRSI"]
    .notna()
)


tr_conversion_mismatches = 0


for mg_dl, released_si in zip(
    df.loc[
        valid_tr_mask,
        "LBXTR"
    ],
    df.loc[
        valid_tr_mask,
        "LBDTRSI"
    ]
):

    calculated = (
        Decimal(
            str(int(mg_dl))
        )
        *
        Decimal("0.01129")
    )


    calculated_3dp = calculated.quantize(
        Decimal("0.001"),
        rounding=ROUND_HALF_UP
    )


    released_3dp = Decimal(
        str(float(released_si))
    ).quantize(
        Decimal("0.001")
    )


    if calculated_3dp != released_3dp:

        tr_conversion_mismatches += 1


print(
    "\n--- TRIGLYCERIDE SI CONVERSION CHECK ---"
)

print(
    "Records checked:",
    f"{int(valid_tr_mask.sum()):,}"
)

print(
    "Conversion mismatches:",
    tr_conversion_mismatches
)


if tr_conversion_mismatches != 0:

    raise ValueError(
        "LBDTRSI does not match "
        "LBXTR x 0.01129."
    )


print(
    "PASS: All 2,723 triglyceride SI values "
    "match CDC conversion."
)


# ============================================================
# 22. CDC CHECK - LBDLDL
# ============================================================

#
# LDL cholesterol mg/dL
#
# CDC:
#
#     23-277
#
#     nonmissing = 2,699
#     missing    =   492
#

ldl_nonmissing = int(
    df["LBDLDL"]
    .notna()
    .sum()
)


ldl_missing = int(
    df["LBDLDL"]
    .isna()
    .sum()
)


ldl_range = int(
    df["LBDLDL"]
    .between(
        23,
        277
    )
    .sum()
)


ldl_min = int(
    df["LBDLDL"]
    .min()
)


ldl_max = int(
    df["LBDLDL"]
    .max()
)


print(
    "\n--- LBDLDL ---"
)

print(
    "Nonmissing:",
    f"{ldl_nonmissing:,}"
)

print(
    "Missing:",
    f"{ldl_missing:,}"
)

print(
    "Minimum:",
    ldl_min
)

print(
    "Maximum:",
    ldl_max
)


if ldl_nonmissing != 2699:

    raise ValueError(
        "LBDLDL nonmissing count differs."
    )


if ldl_missing != 492:

    raise ValueError(
        "LBDLDL missing count differs."
    )


if ldl_range != 2699:

    raise ValueError(
        "LBDLDL contains values outside 23-277."
    )


if ldl_min != 23:

    raise ValueError(
        "LBDLDL minimum differs."
    )


if ldl_max != 277:

    raise ValueError(
        "LBDLDL maximum differs."
    )


print(
    "PASS: LBDLDL matches CDC range/count."
)


# ============================================================
# 23. CDC CHECK - LBDLDLSI
# ============================================================

#
# LDL cholesterol mmol/L
#
# CDC:
#
#     0.595-7.163
#
#     nonmissing = 2,699
#     missing    =   492
#

ldlsi_nonmissing = int(
    df["LBDLDLSI"]
    .notna()
    .sum()
)


ldlsi_missing = int(
    df["LBDLDLSI"]
    .isna()
    .sum()
)


ldlsi_range = int(
    df["LBDLDLSI"]
    .between(
        0.595,
        7.163
    )
    .sum()
)


ldlsi_min = float(
    df["LBDLDLSI"]
    .min()
)


ldlsi_max = float(
    df["LBDLDLSI"]
    .max()
)


print(
    "\n--- LBDLDLSI ---"
)

print(
    "Nonmissing:",
    f"{ldlsi_nonmissing:,}"
)

print(
    "Missing:",
    f"{ldlsi_missing:,}"
)

print(
    "Minimum:",
    repr(ldlsi_min)
)

print(
    "Maximum:",
    repr(ldlsi_max)
)


if ldlsi_nonmissing != 2699:

    raise ValueError(
        "LBDLDLSI nonmissing count differs."
    )


if ldlsi_missing != 492:

    raise ValueError(
        "LBDLDLSI missing count differs."
    )


if ldlsi_range != 2699:

    raise ValueError(
        "LBDLDLSI contains values outside CDC range."
    )


if ldlsi_min != 0.595:

    raise ValueError(
        "LBDLDLSI minimum differs."
    )


if ldlsi_max != 7.163:

    raise ValueError(
        "LBDLDLSI maximum differs."
    )


print(
    "PASS: LBDLDLSI matches CDC range/count."
)


# ============================================================
# 24. LDL / LDL-SI MISSING-MASK CHECK
# ============================================================

ldl_missing_mask = (
    df["LBDLDL"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


ldlsi_missing_mask = (
    df["LBDLDLSI"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


ldl_mask_mismatches = int(
    np.sum(
        ldl_missing_mask
        !=
        ldlsi_missing_mask
    )
)


print(
    "\n--- LDL MISSING MASK CHECK ---"
)

print(
    "Mismatches:",
    ldl_mask_mismatches
)


if ldl_mask_mismatches != 0:

    raise ValueError(
        "LBDLDL and LBDLDLSI missing "
        "positions differ."
    )


print(
    "PASS: LDL / LDL-SI missing "
    "positions match exactly."
)


# ============================================================
# 25. CDC LDL-SI CONVERSION CHECK
# ============================================================

#
# CDC:
#
#     LBDLDLSI = LBDLDL * 0.02586
#
#
# Released values are represented to 3 decimals.
#
# Validation ONLY.
#
# We do not overwrite LBDLDLSI.
#

valid_ldl_mask = (
    df["LBDLDL"]
    .notna()
    &
    df["LBDLDLSI"]
    .notna()
)


ldl_conversion_mismatches = 0


for mg_dl, released_si in zip(
    df.loc[
        valid_ldl_mask,
        "LBDLDL"
    ],
    df.loc[
        valid_ldl_mask,
        "LBDLDLSI"
    ]
):

    calculated = (
        Decimal(
            str(int(mg_dl))
        )
        *
        Decimal("0.02586")
    )


    calculated_3dp = calculated.quantize(
        Decimal("0.001"),
        rounding=ROUND_HALF_UP
    )


    released_3dp = Decimal(
        str(float(released_si))
    ).quantize(
        Decimal("0.001")
    )


    if calculated_3dp != released_3dp:

        ldl_conversion_mismatches += 1


print(
    "\n--- LDL SI CONVERSION CHECK ---"
)

print(
    "Records checked:",
    f"{int(valid_ldl_mask.sum()):,}"
)

print(
    "Conversion mismatches:",
    ldl_conversion_mismatches
)


if ldl_conversion_mismatches != 0:

    raise ValueError(
        "LBDLDLSI does not match "
        "LBDLDL x 0.02586."
    )


print(
    "PASS: All 2,699 LDL SI values "
    "match CDC conversion."
)


# ============================================================
# 26. LDL / TRIGLYCERIDE THRESHOLD CHECK
# ============================================================

#
# CDC states that Friedewald LDL is valid when:
#
#     triglycerides <= 400 mg/dL
#
#
# Exact attachment:
#
#     LBXTR <= 400  = 2,699
#     LBXTR > 400   =    24
#     LBXTR missing =   468
#
#
# LBDLDL exists for ALL 2,699 records where TR <= 400
# and is missing for all 24 where TR > 400.
#

tr_le_400 = int(
    df["LBXTR"]
    .le(400)
    .sum()
)


tr_gt_400 = int(
    df["LBXTR"]
    .gt(400)
    .sum()
)


tr_missing_count = int(
    df["LBXTR"]
    .isna()
    .sum()
)


ldl_present_le400 = int(
    (
        df["LBXTR"].le(400)
        &
        df["LBDLDL"].notna()
    )
    .sum()
)


ldl_present_gt400 = int(
    (
        df["LBXTR"].gt(400)
        &
        df["LBDLDL"].notna()
    )
    .sum()
)


ldl_missing_gt400 = int(
    (
        df["LBXTR"].gt(400)
        &
        df["LBDLDL"].isna()
    )
    .sum()
)


print(
    "\n--- LDL / TRIGLYCERIDE THRESHOLD ---"
)

print(
    "LBXTR <= 400:",
    tr_le_400
)

print(
    "LBXTR > 400:",
    tr_gt_400
)

print(
    "LBXTR missing:",
    tr_missing_count
)

print(
    "LDL present when TR <= 400:",
    ldl_present_le400
)

print(
    "LDL present when TR > 400:",
    ldl_present_gt400
)

print(
    "LDL missing when TR > 400:",
    ldl_missing_gt400
)


if tr_le_400 != 2699:

    raise ValueError(
        "Unexpected LBXTR <=400 count."
    )


if tr_gt_400 != 24:

    raise ValueError(
        "Unexpected LBXTR >400 count."
    )


if tr_missing_count != 468:

    raise ValueError(
        "Unexpected LBXTR missing count."
    )


if ldl_present_le400 != 2699:

    raise ValueError(
        "Not all TR<=400 records have LDL."
    )


if ldl_present_gt400 != 0:

    raise ValueError(
        "LDL unexpectedly populated when TR>400."
    )


if ldl_missing_gt400 != 24:

    raise ValueError(
        "Expected all 24 TR>400 records "
        "to have missing LDL."
    )


print(
    "PASS: LDL availability exactly follows "
    "the documented triglyceride threshold."
)


# ============================================================
# 27. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original ordinary zero count = 0.
#
# Therefore after correction the ONLY zeros should be:
#
#     WTSAF2YR = 448
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


EXPECTED_ZERO_COUNTS = {
    "WTSAF2YR": 448
}


if final_zero_counts != EXPECTED_ZERO_COUNTS:

    raise ValueError(
        "Final zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_ZERO_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final zero pattern contains "
    "exactly 448 WTSAF2YR zeros."
)


# ============================================================
# 28. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Only:
#
#     WTSAF2YR tiny artifact -> 0
#
# is permitted.
#
# Everything else must remain numerically identical.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 6 variables match "
    "the exact corrected XPT values."
)


# ============================================================
# 29. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# Results:
#
# SEQN:
#     83733.0 -> 83733
#
# LBXTR:
#     147.0 -> 147
#
# LBDLDL:
#     173.0 -> 173
#
#
# Continuous values remain float:
#
# WTSAF2YR
# LBDTRSI
# LBDLDLSI
#
#
# Same fidelity rule applies to your Dietary decimals:
#
#     AS IS
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 30. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 31. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 6 TRIGLY_I variables."
)


# ============================================================
# 32. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 33. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 34. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 35. .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# Do NOT perform this against:
#
#     WTSAF2YR
#     LBDTRSI
#     LBDLDLSI
#
#
# Only:
#
#     SEQN
#     LBXTR
#     LBDLDL
#
# should lose unnecessary ".0".
#

print(
    "\n--- INTEGER FIELD .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' in integer fields:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' in "
    "SEQN, LBXTR or LBDLDL."
)


# ============================================================
# 36. LEGITIMATE DECIMAL .0 TOKENS
# ============================================================

#
# INFORMATIONAL + FIDELITY CHECK.
#
# Attached file:
#
# WTSAF2YR
#     448 zero weights -> "0.0"
#
# LBDTRSI
#     0 whole-looking values
#
# LBDLDLSI
#     23 legitimate whole-looking values
#

print(
    "\n--- LEGITIMATE DECIMAL .0 TOKENS ---"
)


decimal_dot_zero_counts = {}


for col in DECIMAL_COLUMNS:

    count = int(
        (
            raw_csv_df[col]
            .ne("")
            &
            raw_csv_df[col]
            .str.endswith(".0")
        )
        .sum()
    )


    decimal_dot_zero_counts[col] = count


    print(
        f"{col}:",
        f"{count:,}"
    )


if decimal_dot_zero_counts["WTSAF2YR"] != 448:

    raise ValueError(
        "Unexpected WTSAF2YR '.0' token count."
    )


if decimal_dot_zero_counts["LBDTRSI"] != 0:

    raise ValueError(
        "Unexpected LBDTRSI '.0' token count."
    )


if decimal_dot_zero_counts["LBDLDLSI"] != 23:

    raise ValueError(
        "Unexpected LBDLDLSI '.0' token count."
    )


print(
    "PASS: Legitimate decimal '.0' values preserved."
)


# ============================================================
# 37. EXACT CONTINUOUS-VALUE FIDELITY
# ============================================================

#
# rtol = 0
# atol = 0
#
# No numerical differences accepted.
#

print(
    "\n--- EXACT DECIMAL FIDELITY ---"
)


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Continuous-value fidelity failed:\n"
        + str(
            decimal_failures
        )
    )


print(
    "PASS: WTSAF2YR, LBDTRSI and LBDLDLSI "
    "preserved exactly."
)


# ============================================================
# 38. FINAL WTSAF2YR PRECISION CHECK
# ============================================================

final_weight = pd.to_numeric(
    raw_csv_df["WTSAF2YR"]
    .where(
        raw_csv_df["WTSAF2YR"] != "",
        np.nan
    ),
    errors="coerce"
)


positive_final_weight = (
    final_weight[
        final_weight > 0
    ]
)


final_weight_min = float(
    positive_final_weight.min()
)


final_weight_max = float(
    positive_final_weight.max()
)


print(
    "\n--- FINAL WTSAF2YR PRECISION ---"
)

print(
    "Minimum:",
    repr(final_weight_min)
)

print(
    "Maximum:",
    repr(final_weight_max)
)


if final_weight_min != 13612.331812498:

    raise ValueError(
        "WTSAF2YR minimum changed during export."
    )


if final_weight_max != 521632.1858299:

    raise ValueError(
        "WTSAF2YR maximum changed during export."
    )


print(
    "PASS: WTSAF2YR full XPT precision preserved."
)


# ============================================================
# 39. FINAL LAB RANGE CHECK
# ============================================================

final_tr = pd.to_numeric(
    raw_csv_df["LBXTR"]
    .where(
        raw_csv_df["LBXTR"] != "",
        np.nan
    ),
    errors="coerce"
)


final_trsi = pd.to_numeric(
    raw_csv_df["LBDTRSI"]
    .where(
        raw_csv_df["LBDTRSI"] != "",
        np.nan
    ),
    errors="coerce"
)


final_ldl = pd.to_numeric(
    raw_csv_df["LBDLDL"]
    .where(
        raw_csv_df["LBDLDL"] != "",
        np.nan
    ),
    errors="coerce"
)


final_ldlsi = pd.to_numeric(
    raw_csv_df["LBDLDLSI"]
    .where(
        raw_csv_df["LBDLDLSI"] != "",
        np.nan
    ),
    errors="coerce"
)


if final_tr.min() != 14:

    raise ValueError(
        "Final LBXTR minimum changed."
    )


if final_tr.max() != 2141:

    raise ValueError(
        "Final LBXTR maximum changed."
    )


if final_trsi.min() != 0.158:

    raise ValueError(
        "Final LBDTRSI minimum changed."
    )


if final_trsi.max() != 24.172:

    raise ValueError(
        "Final LBDTRSI maximum changed."
    )


if final_ldl.min() != 23:

    raise ValueError(
        "Final LBDLDL minimum changed."
    )


if final_ldl.max() != 277:

    raise ValueError(
        "Final LBDLDL maximum changed."
    )


if final_ldlsi.min() != 0.595:

    raise ValueError(
        "Final LBDLDLSI minimum changed."
    )


if final_ldlsi.max() != 7.163:

    raise ValueError(
        "Final LBDLDLSI maximum changed."
    )


print(
    "PASS: Final triglyceride/LDL ranges preserved."
)


# ============================================================
# 40. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != {
    "WTSAF2YR": 448
}:

    raise ValueError(
        "Final CSV zero pattern differs."
    )


print(
    "PASS: Final CSV contains exactly "
    "448 validated WTSAF2YR zeros."
)


# ============================================================
# 41. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 42. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83733:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93697:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 43. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "TRIGLY_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Laboratory - "
    "LDL Cholesterol & Triglycerides"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Tiny artifact variable:",
    "WTSAF2YR"
)


print(
    "Validated zero restorations:",
    "448"
)


print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)


print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


print(
    "WTSAF2YR positive:",
    f"{weight_positive:,}"
)


print(
    "WTSAF2YR zero:",
    f"{weight_zero:,}"
)


print(
    "WTSAF2YR exact minimum:",
    repr(weight_min)
)


print(
    "WTSAF2YR exact maximum:",
    repr(weight_max)
)


print(
    "LBXTR nonmissing:",
    f"{tr_nonmissing:,}"
)


print(
    "LBXTR missing:",
    f"{tr_missing:,}"
)


print(
    "LBXTR range:",
    f"{tr_min}-{tr_max} mg/dL"
)


print(
    "LBDTRSI range:",
    f"{trsi_min}-{trsi_max} mmol/L"
)


print(
    "LBDLDL nonmissing:",
    f"{ldl_nonmissing:,}"
)


print(
    "LBDLDL missing:",
    f"{ldl_missing:,}"
)


print(
    "LBDLDL range:",
    f"{ldl_min}-{ldl_max} mg/dL"
)


print(
    "LBDLDLSI range:",
    f"{ldlsi_min}-{ldlsi_max} mmol/L"
)


print(
    "TR >400 records:",
    tr_gt_400
)


print(
    "TR >400 with LDL populated:",
    ldl_present_gt400
)


print(
    "Triglyceride SI conversion mismatches:",
    tr_conversion_mismatches
)


print(
    "LDL SI conversion mismatches:",
    ldl_conversion_mismatches
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Continuous-value differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0' in integer fields:",
    "0 expected"
)


print(
    "Continuous '.0' values:",
    "ALLOWED / PRESERVED"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "WTSAF2YR, LBDTRSI and LBDLDLSI "
    "remain genuine continuous decimal variables."
)


print(
    "No continuous Laboratory value was "
    "rounded or rewritten."
)

NHANES 2015-2016 TRIGLY_I XPT -> CSV
Rows: 3,191
Columns: 6
PASS: Structure = 3,191 rows x 6 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: ['WTSAF2YR', 'LBDTRSI', 'LBDLDLSI']
Whole-number fields: ['SEQN', 'LBXTR', 'LBDLDL']

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 448

Tiny values by variable:
WTSAF2YR    448

PASS: Exact 448-value WTSAF2YR tiny-zero pattern validated.

PASS: 448 validated WTSAF2YR values restored to numeric zero.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
WTSAF2YR: 2,743 fractional observations
LBXTR: 0 fractional observations
LBDTRSI: 2,723 fractional observations
LBDLDL: 0 fractional observations
LBDLDLSI: 2,676 fractional observations

PASS: Exact genuine decimal fields detected:
['WTSAF2YR', 'LBDTRSI', 'LBDLDLSI']

--- DECIMAL FIELD DETAILS ---

WTSAF2YR
  Nonmissing: 3,191
  Missing: 0
  Minimum: 0.0
  Maxim